#PatchParkCNN - GROUP 4



* Laura Lugo
* Tatsuta N.
* Leshawn P.
* Esther M.
* Aitua O.



# 01 Environment Setup

This notebook initializes the project environment using object
oriented design.  
It mounts Google Drive and validates the directory structure used across all subsequent notebooks.  
The goal is to maintain a clean and consistent project layout.

In [ ]:
from google.colab import drive
import os

In [ ]:
class EnvironmentManager:
    """
    Handles environment setup for the project:
    - Mounts Google Drive
    - Validates and creates the directory structure
    """

    def __init__(self, base_dir="/content/drive/MyDrive/PatchParkCNN"):
        self.base_dir = base_dir

        # Directories used across the entire project
        self.dirs = {
          "raw":           os.path.join(base_dir, "data_raw"),
          "patches":       os.path.join(base_dir, "data_patches"),
          "patches_sunny": os.path.join(base_dir, "data_patches_sunny"),
          "weights":       os.path.join(base_dir, "weights"),
          "figures":       os.path.join(base_dir, "figures"),
          "csv":           os.path.join(base_dir, "csv"),
          "report":        os.path.join(base_dir, "report"),
          "notebook":      os.path.join(base_dir, "notebook")

        }

    def mount_drive(self):
        """Mounts Google Drive."""
        drive.mount('/content/drive')
        print("Drive mounted.")

    def create_directories(self):
        """Creates all required directories if missing."""
        for name, path in self.dirs.items():
            os.makedirs(path, exist_ok=True)
        print("Project directories are ready.")

    def summary(self):
        """Prints all active paths."""
        print("Project base directory:", self.base_dir)
        print("Subdirectories:")
        for name, path in self.dirs.items():
            print(f" - {name}: {path}")

#02 Data Extraction
PatchParkCNN Project
Advanced Mathematical Concepts for Deep Learning

This notebook prepares the raw inputs for the project using an object oriented design. It extracts slot level images from the original ZIP archive using a cleaned CSV file that maps each image file name to a binary label (free or busy).

In [ ]:
from google.colab import drive
import os

class DriveManager:
    """Handles Google Drive mounting in Colab."""

    def __init__(self, mount_point: str = '/content/drive'):
        self.mount_point = mount_point

    def mount_drive(self):
        """Mounts Google Drive."""
        drive.mount(self.mount_point)
        print("Drive mounted at:", self.mount_point)

In [ ]:
# Project Config

import os

class ProjectConfig:
    """Centralizes paths used by Notebook 02."""

    def __init__(self, base_dir: str = "/content/drive/MyDrive/PatchParkCNN"):
        self.base_dir = base_dir
        self.raw_dir = os.path.join(base_dir, "data_raw")
        self.images_flat_dir = os.path.join(self.raw_dir, "images_flat")

        # Source files
        self.zip_path = os.path.join(self.raw_dir, "CNR-EXT-Patches.zip")
        self.csv_path = os.path.join(self.raw_dir, "CNR.csv")

    def ensure_directories(self):
        os.makedirs(self.raw_dir, exist_ok=True)
        os.makedirs(self.images_flat_dir, exist_ok=True)
        print("Raw directory:", self.raw_dir)
        print("Images output directory:", self.images_flat_dir)

    def summary(self):
        print("Base directory:", self.base_dir)
        print("ZIP path:", self.zip_path)
        print("CSV path:", self.csv_path)
        print("images_flat:", self.images_flat_dir)

In [ ]:
# Image Extractor
# Extracts only images listed in CNR.csv

import zipfile
import shutil
import pandas as pd
import os
from tqdm import tqdm

class ImageExtractor:
    """Extracts specific images from a ZIP based on FileName values in a CSV."""

    def __init__(self, zip_path: str, csv_path: str, output_folder: str,
                 file_col: str = "FileName"):
        self.zip_path = zip_path
        self.csv_path = csv_path
        self.output_folder = output_folder
        self.file_col = file_col

        os.makedirs(self.output_folder, exist_ok=True)

    def _folder_has_files_fast(self, path: str) -> bool:
        """Quickly checks if a folder contains at least one file."""
        try:
            if not os.path.exists(path):
                return False
            with os.scandir(path) as it:
                for entry in it:
                    if entry.is_file():
                        return True
            return False
        except OSError as e:
            print("[WARN] Drive I/O issue while checking output folder")
            print("[WARN]", type(e).__name__, str(e))
            return False

    def _load_filenames(self):
        """Loads target image names from CSV."""
        if not os.path.exists(self.csv_path):
            raise FileNotFoundError(f"CSV not found at: {self.csv_path}")

        df = pd.read_csv(self.csv_path)

        if self.file_col not in df.columns:
            raise ValueError(f"CSV missing required column: {self.file_col}")

        return set(df[self.file_col].astype(str).tolist())

    def extract_images(self, skip_if_exists: bool = True):
        """Extracts images from ZIP into images_flat, with a progress bar."""

        if skip_if_exists and self._folder_has_files_fast(self.output_folder):
            print("images_flat already contains files. Skipping extraction.")
            return

        if not os.path.exists(self.zip_path):
            raise FileNotFoundError(f"ZIP not found at: {self.zip_path}")

        file_names = self._load_filenames()
        print("Number of target images from CSV:", len(file_names))

        with zipfile.ZipFile(self.zip_path, 'r') as zip_ref:

            zip_files = zip_ref.namelist()

            zip_map = {}
            for zf in zip_files:
                base = os.path.basename(zf)
                if base and base not in zip_map:
                    zip_map[base] = zf

            extracted = 0
            missing = 0

            print("\nExtracting images...\n")
            for fname in tqdm(file_names, desc="Processing images", unit="img"):

                zpath = zip_map.get(fname, None)
                if zpath is None:
                    missing += 1
                    continue

                zip_ref.extract(zpath, self.output_folder)

                extracted_path = os.path.join(self.output_folder, zpath)
                final_path = os.path.join(self.output_folder, fname)

                if extracted_path != final_path:
                    os.makedirs(os.path.dirname(final_path), exist_ok=True)
                    shutil.move(extracted_path, final_path)

                extracted += 1

            for root, dirs, files in os.walk(self.output_folder, topdown=False):
                if root == self.output_folder:
                    continue
                if not os.listdir(root):
                    try:
                        os.rmdir(root)
                    except OSError:
                        pass

        print("\n Extraction completed")
        print("Extracted files:", extracted)
        print("Missing files:", missing)
        print("Output folder:", self.output_folder)

In [ ]:
# Main Execution

drive_manager = DriveManager()
drive_manager.mount_drive()

config = ProjectConfig(base_dir="/content/drive/MyDrive/PatchParkCNN")
config.ensure_directories()
config.summary()

extractor = ImageExtractor(
    zip_path=config.zip_path,
    csv_path=config.csv_path,
    output_folder=config.images_flat_dir,
    file_col="FileName"
)

extractor.extract_images(skip_if_exists=True)

print("Notebook 02 finished successfully")

#03 Dataset Builder

In [ ]:
from google.colab import drive
import os
import pandas as pd
import shutil
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [ ]:
class DriveManager:
    def __init__(self, mount_point='/content/drive'):
        self.mount_point = mount_point

    def mount(self):
        drive.mount(self.mount_point, force_remount=True)
        print(f'Drive mounted at: {self.mount_point}')

In [ ]:
class ProjectPaths:
    def __init__(self, base_dir='/content/drive/MyDrive/PatchParkCNN'):
        self.base_dir = base_dir
        self.raw_dir = os.path.join(base_dir, 'data_raw')
        self.images_dir = os.path.join(self.raw_dir, 'images_flat')
        self.output_dir = os.path.join(base_dir, 'data_patches_cnr')

    def ensure_output_dir(self):
        os.makedirs(self.output_dir, exist_ok=True)

    def get_images_dir(self):
        # Drive safe check: do not list all files in large folders
        if os.path.exists(self.images_dir):
            try:
                with os.scandir(self.images_dir) as it:
                    for entry in it:
                        if entry.is_file():
                            print(f"Using images directory: {self.images_dir}")
                            return self.images_dir
            except OSError as e:
                print("[WARN] Drive I O error while checking images_flat, will try fallback.")
                print("[WARN]", type(e).__name__, str(e))

        # fallback to original PATCHES if available
        if hasattr(self, "patches_dir") and os.path.exists(self.patches_dir):
            print(f"Using fallback patches directory: {self.patches_dir}")
            return self.patches_dir

        raise FileNotFoundError("No valid image directory found.")

In [ ]:
class CSVLocator:
    def __init__(self, base_dir, csv_name='CNR.csv'):
        self.base_dir = base_dir
        self.csv_name = csv_name

    def find_csv(self):
        for root, dirs, files in os.walk(self.base_dir):
            if self.csv_name in files:
                csv_path = os.path.join(root, self.csv_name)
                print(f'CSV located at: {csv_path}')
                return csv_path
        raise FileNotFoundError(f'{self.csv_name} not found. Ensure it exists in data_raw.')

In [ ]:
class DatasetBuilder:
    def __init__(self, images_dir, csv_path, output_dir, file_col='FileName', label_col='Status', train_size=0.7, val_size=0.15, random_state=42):
        self.images_dir = images_dir
        self.csv_path = csv_path
        self.output_dir = output_dir
        self.file_col = file_col
        self.label_col = label_col
        self.train_size = train_size
        self.val_size = val_size
        self.random_state = random_state
        os.makedirs(self.output_dir, exist_ok=True)

    def load_dataframe(self):
        df = pd.read_csv(self.csv_path)
        df = df.dropna(subset=[self.file_col, self.label_col])
        df['class_name'] = df[self.label_col].map({0: 'free', 1: 'busy'})
        return df.sample(frac=1, random_state=self.random_state).reset_index(drop=True)

    def split_dataframe(self, df):
        train_df, temp_df = train_test_split(df, train_size=self.train_size, stratify=df['class_name'], random_state=self.random_state)
        val_ratio = self.val_size / (1 - self.train_size)
        val_df, test_df = train_test_split(temp_df, train_size=val_ratio, stratify=temp_df['class_name'], random_state=self.random_state)
        return {'train': train_df, 'val': val_df, 'test': test_df}

    def create_directory_structure(self):
        for split in ['train','val','test']:
            for cls in ['free','busy']:
                os.makedirs(os.path.join(self.output_dir, split, cls), exist_ok=True)

    def find_image_path(self, filename):
        path = os.path.join(self.images_dir, filename)
        return path if os.path.exists(path) else None

    def copy_images(self, split_name, df_split):
        print(f"\n Copying images for split: {split_name}")

        missing_count = 0

        # tqdm = barra de progreso visual
        for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc=f"Copying {split_name}", unit="img"):
            src = self.find_image_path(row[self.file_col])

            if src is None:
                missing_count += 1
                continue

            dst = os.path.join(self.output_dir, split_name, row['class_name'], row[self.file_col])
            shutil.copy(src, dst)

        print(f" {split_name}: {len(df_split) - missing_count} copied, {missing_count} missing\n")


    def build_dataset(self):
        df = self.load_dataframe()
        splits = self.split_dataframe(df)
        self.create_directory_structure()
        for name, df_split in splits.items():
            self.copy_images(name, df_split)
        print(f'Dataset build complete at: {self.output_dir}')
        return splits


In [ ]:
# Execution Section
drive_manager = DriveManager()
drive_manager.mount()

base_path = '/content/drive/MyDrive/PatchParkCNN'
paths = ProjectPaths(base_path)
paths.ensure_output_dir()

images_dir = paths.get_images_dir()
csv_path = CSVLocator(base_path).find_csv()

builder = DatasetBuilder(images_dir, csv_path, paths.output_dir)
dataset_splits = builder.build_dataset()


#04 Exploratory Data Analysis
PatchParkCNN Project
Advanced Mathematical Concepts for Deep Learning

This notebook performs an exploratory data analysis of the slot level dataset. It analyzes the distribution of images across training, validation and test splits, compares free and busy classes, and visualizes representative samples.

The goal is to validate data integrity and justify modeling decisions before CNN training.

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# EDA Manager

class EDAManager:
    """
    Performs exploratory data analysis on a classification dataset
    organized by train, validation and test splits with free and
    busy classes.
    """

    def __init__(self, dataset_dir, figures_dir=None):
        self.dataset_dir = dataset_dir
        self.figures_dir = figures_dir

        if self.figures_dir:
            os.makedirs(self.figures_dir, exist_ok=True)

        self.splits = ["train", "val", "test"]
        self.classes = ["free", "busy"]

    def count_images(self):
        """
        Counts the number of images per split and per class.
        """
        counts = {}

        for split in self.splits:
            counts[split] = {}
            for cls in self.classes:
                path = os.path.join(self.dataset_dir, split, cls)
                counts[split][cls] = len(os.listdir(path))

        return counts

    def plot_class_distribution(self, counts):
        """
        Plots class distribution for each split.
        """
        for split in self.splits:
            free_count = counts[split]["free"]
            busy_count = counts[split]["busy"]

            plt.figure()
            plt.bar(["free", "busy"], [free_count, busy_count])
            plt.title(f"Class distribution in {split} set")
            plt.ylabel("Number of images")

            if self.figures_dir:
                plt.savefig(os.path.join(self.figures_dir, f"class_distribution_{split}.png"))

            plt.show()

    def show_sample_images(self, split="train", samples_per_class=5):
        """
        Displays random sample images for each class.
        """
        plt.figure(figsize=(12, 5))
        idx = 1

        for cls in self.classes:
            path = os.path.join(self.dataset_dir, split, cls)
            images = random.sample(os.listdir(path), samples_per_class)

            for img_name in images:
                img_path = os.path.join(path, img_name)
                img = Image.open(img_path)

                plt.subplot(2, samples_per_class, idx)
                plt.imshow(img)
                plt.axis("off")
                plt.title(cls)
                idx += 1

        plt.suptitle(f"Sample images from {split} set")
        plt.show()

    def summarize(self):
        """
        Prints a summary of the dataset.
        """
        counts = self.count_images()

        print("Dataset summary:\n")
        for split in self.splits:
            total = counts[split]["free"] + counts[split]["busy"]
            print(f"{split.upper()}:")
            print(f"  free: {counts[split]['free']}")
            print(f"  busy: {counts[split]['busy']}")
            print(f"  total: {total}\n")

        return counts

In [ ]:
# Execute EDA

DATASET_DIR = "/content/drive/MyDrive/PatchParkCNN/data_patches_cnr"
FIGURES_DIR = "/content/drive/MyDrive/PatchParkCNN/figures"

eda = EDAManager(
    dataset_dir=DATASET_DIR,
    figures_dir=FIGURES_DIR
)

counts = eda.summarize()
eda.plot_class_distribution(counts)
eda.show_sample_images(split="train", samples_per_class=5)

#05 Convolutional Neural Network Training

Convolutional Neural Network Training
PatchParkCNN Project
Advanced Mathematical Concepts for Deep Learning

We trains a convolutional neural network from scratch for patch level parking slot classification (free vs busy)

In [ ]:
from google.colab import drive
import os
import json
import time
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


In [ ]:
class DriveManager:
    """Handles Google Drive mounting in Colab."""

    def __init__(self, mount_point: str = "/content/drive"):
        self.mount_point = mount_point

    def mount_drive(self, force_remount: bool = False) -> None:
        drive.mount(self.mount_point, force_remount=force_remount)
        print("Drive mounted at:", self.mount_point)


@dataclass
class TrainingConfig:
    """Stores training configuration values."""

    image_height: int = 128
    image_width: int = 128
    batch_size: int = 64
    seed: int = 42
    epochs: int = 30
    learning_rate: float = 0.001
    dropout_rate: float = 0.35
    l2_weight: float = 0.0001
    patience_early_stop: int = 6
    patience_lr: int = 3


class ProjectConfig:
    """Centralizes paths used by the project."""

    def __init__(self, base_dir: str = "/content/drive/MyDrive/PatchParkCNN"):
        self.base_dir = base_dir

        self.data_patches_dir = os.path.join(self.base_dir, "data_patches_cnr")
        self.train_dir = os.path.join(self.data_patches_dir, "train")
        self.val_dir = os.path.join(self.data_patches_dir, "val")
        self.test_dir = os.path.join(self.data_patches_dir, "test")

        self.figures_dir = os.path.join(self.base_dir, "figures")
        self.weights_dir = os.path.join(self.base_dir, "weights")
        self.csv_dir = os.path.join(self.base_dir, "csv")
        self.report_dir = os.path.join(self.base_dir, "report")

        self.tensorboard_dir = os.path.join(self.report_dir, "tensorboard_logs")
        self.run_id = time.strftime("%Y%m%d_%H%M%S")

    def ensure_directories(self) -> None:
        for path in [self.figures_dir, self.weights_dir, self.csv_dir, self.report_dir, self.tensorboard_dir]:
            os.makedirs(path, exist_ok=True)

        print("Project directories are ready")
        print("Base directory:", self.base_dir)
        print("Train directory:", self.train_dir)
        print("Validation directory:", self.val_dir)
        print("Test directory:", self.test_dir)
        print("Figures directory:", self.figures_dir)
        print("Weights directory:", self.weights_dir)
        print("CSV directory:", self.csv_dir)
        print("Report directory:", self.report_dir)

    def assert_dataset_structure(self) -> None:
        required = [
            self.train_dir,
            os.path.join(self.train_dir, "free"),
            os.path.join(self.train_dir, "busy"),
            self.val_dir,
            os.path.join(self.val_dir, "free"),
            os.path.join(self.val_dir, "busy"),
        ]
        missing = [p for p in required if not os.path.exists(p)]
        if missing:
            raise FileNotFoundError("Missing dataset folders:\n" + "\n".join(missing))

    def artifact_paths(self) -> dict:
        return {
            "best_model_path": os.path.join(self.weights_dir, f"cnn_best_{self.run_id}.keras"),
            "final_model_path": os.path.join(self.weights_dir, f"cnn_final_{self.run_id}.keras"),
            "history_csv_path": os.path.join(self.csv_dir, f"history_{self.run_id}.csv"),
            "training_plot_path": os.path.join(self.figures_dir, f"training_curves_{self.run_id}.png"),
            "config_json_path": os.path.join(self.report_dir, f"training_config_{self.run_id}.json"),
            "model_summary_path": os.path.join(self.report_dir, f"model_summary_{self.run_id}.txt"),
            "tensorboard_run_dir": os.path.join(self.tensorboard_dir, f"run_{self.run_id}"),
        }


class SystemInspector:
    """Collects environment information and prints it for reporting."""

    def get_device_summary(self) -> dict:
        gpus = tf.config.list_physical_devices("GPU")
        cpus = tf.config.list_physical_devices("CPU")
        return {
            "tensorflow_version": tf.__version__,
            "num_gpus": len(gpus),
            "gpus": [gpu.name for gpu in gpus],
            "num_cpus": len(cpus),
        }

    def print_device_summary(self) -> None:
        summary = self.get_device_summary()
        print("TensorFlow version:", summary["tensorflow_version"])
        print("Number of GPUs:", summary["num_gpus"])
        if summary["num_gpus"] > 0:
            for i, name in enumerate(summary["gpus"], start=1):
                print("GPU", i, ":", name)
        print("Number of CPUs:", summary["num_cpus"])


class ReproducibilityManager:
    """Sets random seeds for reproducible training."""

    def __init__(self, seed: int = 42):
        self.seed = seed

    def set_all_seeds(self) -> None:
        os.environ["PYTHONHASHSEED"] = str(self.seed)
        random.seed(self.seed)
        np.random.seed(self.seed)
        tf.random.set_seed(self.seed)
        print("Seeds set to:", self.seed)


In [ ]:
class DatasetLoader:
    """Loads training and validation datasets using a directory structure."""

    def __init__(self, train_dir: str, val_dir: str, config: TrainingConfig):
        self.train_dir = train_dir
        self.val_dir = val_dir
        self.config = config
        self.class_names = None
        self.train_ds = None
        self.val_ds = None

    def load(self):
        image_size = (self.config.image_height, self.config.image_width)

      train_ds = tf.keras.utils.image_dataset_from_directory(
          self.train_dir,
          labels="inferred",
          label_mode="binary",
          image_size=image_size,
          batch_size=self.config.batch_size,
          shuffle=True,
          seed=self.config.seed,
      )

        val_ds = tf.keras.utils.image_dataset_from_directory(
            self.val_dir,
            labels="inferred",
            label_mode="binary",
            image_size=image_size,
            batch_size=self.config.batch_size,
            shuffle=False,
        )

        self.class_names = train_ds.class_names
        self.train_ds = train_ds
        self.val_ds = val_ds

        print("Detected class names:", self.class_names)
        return train_ds, val_ds

    def configure_performance(self):
        if self.train_ds is None or self.val_ds is None:
            raise RuntimeError("Datasets are not loaded")

        autotune = tf.data.AUTOTUNE
        self.train_ds = self.train_ds.cache().prefetch(buffer_size=autotune)
        self.val_ds = self.val_ds.cache().prefetch(buffer_size=autotune)
        return self.train_ds, self.val_ds

    def compute_class_weights(self) -> dict:
        if self.class_names is None:
            raise RuntimeError("Class names are not available")

        counts = {}
        for class_name in self.class_names:
            class_path = os.path.join(self.train_dir, class_name)
            counts[class_name] = len([f for f in os.listdir(class_path) if os.path.isfile(os.path.join(class_path, f))])

        total = sum(counts.values())
        class_weights = {}

        for idx, class_name in enumerate(self.class_names):
            class_count = max(counts[class_name], 1)
            class_weights[idx] = total / (len(self.class_names) * class_count)

        print("Train class counts:", counts)
        print("Computed class weights:", class_weights)
        return class_weights


In [ ]:
class ModelFactory:
    """Builds a convolutional neural network from scratch."""

    def __init__(self, config: TrainingConfig):
        self.config = config

    def build(self) -> keras.Model:
        reg = keras.regularizers.l2(self.config.l2_weight)
        input_shape = (self.config.image_height, self.config.image_width, 3)

        augmentation = keras.Sequential(
            [
                layers.RandomFlip("horizontal", seed=self.config.seed),
                layers.RandomRotation(0.08, seed=self.config.seed),
                layers.RandomZoom(0.10, seed=self.config.seed),
                layers.RandomContrast(0.10, seed=self.config.seed),
            ],
            name="augmentation",
        )

        inputs = keras.Input(shape=input_shape, name="input_image")
        x = augmentation(inputs)
        x = layers.Rescaling(1.0 / 255.0, name="rescale")(x)

        x = self._conv_block(x, 32, reg, "block1")
        x = self._conv_block(x, 64, reg, "block2")
        x = self._conv_block(x, 128, reg, "block3")

        x = layers.Conv2D(256, (3, 3), padding="same", kernel_regularizer=reg, name="conv4")(x)
        x = layers.BatchNormalization(name="bn4")(x)
        x = layers.Activation("relu", name="relu4")(x)
        x = layers.MaxPooling2D((2, 2), name="pool4")(x)
        x = layers.Dropout(self.config.dropout_rate, name="dropout4")(x)

        x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
        x = layers.Dense(128, kernel_regularizer=reg, name="dense1")(x)
        x = layers.BatchNormalization(name="bn_dense1")(x)
        x = layers.Activation("relu", name="relu_dense1")(x)
        x = layers.Dropout(self.config.dropout_rate, name="dropout_dense1")(x)

        outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

        model = keras.Model(inputs=inputs, outputs=outputs, name="PatchParkCNNModel")
        return model

    def _conv_block(self, x, filters: int, reg, block_name: str):
        x = layers.Conv2D(filters, (3, 3), padding="same", kernel_regularizer=reg, name=f"{block_name}_conv1")(x)
        x = layers.BatchNormalization(name=f"{block_name}_bn1")(x)
        x = layers.Activation("relu", name=f"{block_name}_relu1")(x)

        x = layers.Conv2D(filters, (3, 3), padding="same", kernel_regularizer=reg, name=f"{block_name}_conv2")(x)
        x = layers.BatchNormalization(name=f"{block_name}_bn2")(x)
        x = layers.Activation("relu", name=f"{block_name}_relu2")(x)

        x = layers.MaxPooling2D((2, 2), name=f"{block_name}_pool")(x)
        x = layers.Dropout(self.config.dropout_rate, name=f"{block_name}_dropout")(x)
        return x


In [ ]:
class Trainer:
    """Compiles and trains the model and saves artifacts."""

    def __init__(self, model: keras.Model, config: TrainingConfig, artifact_paths: dict):
        self.model = model
        self.config = config
        self.artifact_paths = artifact_paths
        self.history = None

    def compile(self) -> None:
        optimizer = keras.optimizers.Adam(learning_rate=self.config.learning_rate)
        loss_fn = keras.losses.BinaryCrossentropy()

        self.model.compile(
            optimizer=optimizer,
            loss=loss_fn,
            metrics=[
                keras.metrics.BinaryAccuracy(name="accuracy"),
                keras.metrics.Precision(name="precision"),
                keras.metrics.Recall(name="recall"),
                keras.metrics.AUC(name="auc"),
            ],
        )

    def get_callbacks(self) -> list:
        callbacks = []

        checkpoint = keras.callbacks.ModelCheckpoint(
            filepath=self.artifact_paths["best_model_path"],
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            save_weights_only=False,
            verbose=1,
        )
        callbacks.append(checkpoint)

        early_stop = keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=self.config.patience_early_stop,
            restore_best_weights=True,
            verbose=1,
        )
        callbacks.append(early_stop)

        reduce_lr = keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=self.config.patience_lr,
            min_lr=0.000001,
            verbose=1,
        )
        callbacks.append(reduce_lr)

        tensorboard = keras.callbacks.TensorBoard(
            log_dir=self.artifact_paths["tensorboard_run_dir"],
            histogram_freq=1,
            write_graph=True,
            update_freq="epoch",
        )
        callbacks.append(tensorboard)

        return callbacks

    def fit(self, train_ds, val_ds, class_weight: dict = None):
        callbacks = self.get_callbacks()

        history = self.model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=self.config.epochs,
            callbacks=callbacks,
            class_weight=class_weight,
            verbose=1,
        )

        self.history = history
        return history

    def save_final_model(self) -> None:
        self.model.save(self.artifact_paths["final_model_path"])
        print("Final model saved to:", self.artifact_paths["final_model_path"])

    def save_history_csv(self) -> None:
        if self.history is None:
            raise RuntimeError("History is not available")

        hist_df = pd.DataFrame(self.history.history)
        hist_df.to_csv(self.artifact_paths["history_csv_path"], index=False)
        print("Training history saved to:", self.artifact_paths["history_csv_path"])


In [ ]:
class ReportWriter:
    """Writes configuration and model summary to disk."""

    def __init__(self, artifact_paths: dict):
        self.artifact_paths = artifact_paths

    def save_config(self, config: TrainingConfig, device_summary: dict) -> None:
        payload = {
            "training_config": config.__dict__,
            "device_summary": device_summary,
            "timestamp": time.strftime("%Y%m%d_%H%M%S"),
        }
        with open(self.artifact_paths["config_json_path"], "w") as f:
            json.dump(payload, f, indent=2)
        print("Training configuration saved to:", self.artifact_paths["config_json_path"])

    def save_model_summary(self, model: keras.Model) -> None:
        lines = []
        model.summary(print_fn=lambda x: lines.append(x))
        summary_text = "\n".join(lines)
        with open(self.artifact_paths["model_summary_path"], "w") as f:
            f.write(summary_text)
        print("Model summary saved to:", self.artifact_paths["model_summary_path"])


In [ ]:
class HistoryPlotter:
    """Plots training and validation curves."""

    def __init__(self, artifact_paths: dict):
        self.artifact_paths = artifact_paths

    def plot_and_save(self, history) -> None:
        hist = history.history

        fig = plt.figure(figsize=(12, 8))

        ax1 = fig.add_subplot(2, 1, 1)
        ax1.plot(hist.get("loss", []), label="train_loss")
        ax1.plot(hist.get("val_loss", []), label="val_loss")
        ax1.set_title("Loss")
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
        ax1.legend()

        ax2 = fig.add_subplot(2, 1, 2)
        ax2.plot(hist.get("accuracy", []), label="train_accuracy")
        ax2.plot(hist.get("val_accuracy", []), label="val_accuracy")
        ax2.set_title("Accuracy")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Accuracy")
        ax2.legend()

        plt.tight_layout()
        plt.savefig(self.artifact_paths["training_plot_path"], dpi=200)
        plt.show()

        print("Training curves saved to:", self.artifact_paths["training_plot_path"])


In [ ]:
def main() -> None:
    drive_manager = DriveManager()
    drive_manager.mount_drive(force_remount=False)

    project = ProjectConfig(base_dir="/content/drive/MyDrive/PatchParkCNN")
    project.ensure_directories()
    project.assert_dataset_structure()
    artifacts = project.artifact_paths()

    system_inspector = SystemInspector()
    system_inspector.print_device_summary()
    device_summary = system_inspector.get_device_summary()

    training_config = TrainingConfig()

    reproducibility = ReproducibilityManager(seed=training_config.seed)
    reproducibility.set_all_seeds()

    dataset_loader = DatasetLoader(
        train_dir=project.train_dir,
        val_dir=project.val_dir,
        config=training_config,
    )
    train_ds, val_ds = dataset_loader.load()
    train_ds, val_ds = dataset_loader.configure_performance()

    class_weight = dataset_loader.compute_class_weights()

    model_factory = ModelFactory(config=training_config)
    model = model_factory.build()

    report_writer = ReportWriter(artifact_paths=artifacts)
    report_writer.save_config(training_config, device_summary)
    report_writer.save_model_summary(model)

    trainer = Trainer(model=model, config=training_config, artifact_paths=artifacts)
    trainer.compile()

    history = trainer.fit(train_ds=train_ds, val_ds=val_ds, class_weight=class_weight)

    trainer.save_history_csv()
    trainer.save_final_model()

    plotter = HistoryPlotter(artifact_paths=artifacts)
    plotter.plot_and_save(history)

    print("Notebook 5 completed successfully")


if __name__ == "__main__":
    main()


#06 Model Evaluation and Error Analysis
PatchParkCNN Project
Advanced Mathematical Concepts for Deep Learning

We evaluate a convolutional neural network trained for patch level parking slot classification (free vs busy). The notebook loads the most recent saved model, computes metrics on the test set, produces error analysis, and saves evaluation artifacts.

In [ ]:
from google.colab import drive
import os
import json
import time
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score


In [ ]:
class DriveManager:
    """Handles Google Drive mounting in Colab."""

    def __init__(self, mount_point: str = "/content/drive"):
        self.mount_point = mount_point

    def mount_drive(self, force_remount: bool = False) -> None:
        drive.mount(self.mount_point, force_remount=force_remount)
        print("Drive mounted at:", self.mount_point)


@dataclass
class EvaluationConfig:
    """Stores evaluation configuration values."""

    image_height: int = 128
    image_width: int = 128
    batch_size: int = 64
    seed: int = 42
    threshold: float = 0.5
    max_error_samples: int = 24


class ProjectConfig:
    """Centralizes paths used by the project."""

    def __init__(self, base_dir: str = "/content/drive/MyDrive/PatchParkCNN"):
        self.base_dir = base_dir

        self.data_patches_dir = os.path.join(self.base_dir, "data_patches_cnr")
        self.train_dir = os.path.join(self.data_patches_dir, "train")
        self.val_dir = os.path.join(self.data_patches_dir, "val")
        self.test_dir = os.path.join(self.data_patches_dir, "test")

        self.figures_dir = os.path.join(self.base_dir, "figures")
        self.weights_dir = os.path.join(self.base_dir, "weights")
        self.csv_dir = os.path.join(self.base_dir, "csv")
        self.report_dir = os.path.join(self.base_dir, "report")

        self.run_id = time.strftime("%Y%m%d_%H%M%S")

    def ensure_directories(self) -> None:
        for path in [self.figures_dir, self.weights_dir, self.csv_dir, self.report_dir]:
            os.makedirs(path, exist_ok=True)
        print("Project directories are ready")

    def assert_test_structure(self) -> None:
        required = [
            self.test_dir,
            os.path.join(self.test_dir, "free"),
            os.path.join(self.test_dir, "busy"),
        ]
        missing = [p for p in required if not os.path.exists(p)]
        if missing:
            raise FileNotFoundError("Missing test folders:\n" + "\n".join(missing))

    def artifact_paths(self) -> dict:
        return {
            "predictions_csv_path": os.path.join(self.csv_dir, f"predictions_{self.run_id}.csv"),
            "metrics_json_path": os.path.join(self.report_dir, f"evaluation_metrics_{self.run_id}.json"),
            "classification_report_path": os.path.join(self.report_dir, f"classification_report_{self.run_id}.txt"),
            "confusion_matrix_csv_path": os.path.join(self.csv_dir, f"confusion_matrix_{self.run_id}.csv"),
            "confusion_matrix_plot_path": os.path.join(self.figures_dir, f"confusion_matrix_{self.run_id}.png"),
            "error_grid_plot_path": os.path.join(self.figures_dir, f"error_samples_{self.run_id}.png"),
            "model_used_path": os.path.join(self.report_dir, f"model_used_{self.run_id}.txt"),
        }


class SystemInspector:
    """Collects environment information and prints it for reporting."""

    def get_device_summary(self) -> dict:
        gpus = tf.config.list_physical_devices("GPU")
        cpus = tf.config.list_physical_devices("CPU")
        return {
            "tensorflow_version": tf.__version__,
            "num_gpus": len(gpus),
            "gpus": [gpu.name for gpu in gpus],
            "num_cpus": len(cpus),
        }

    def print_device_summary(self) -> None:
        summary = self.get_device_summary()
        print("TensorFlow version:", summary["tensorflow_version"])
        print("Number of GPUs:", summary["num_gpus"])
        print("Number of CPUs:", summary["num_cpus"])


class DatasetLoader:
    """Loads test dataset using directory structure."""

    def __init__(self, test_dir: str, config: EvaluationConfig):
        self.test_dir = test_dir
        self.config = config
        self.class_names = None

    def load(self):
        image_size = (self.config.image_height, self.config.image_width)
        test_ds = tf.keras.utils.image_dataset_from_directory(
            self.test_dir,
            labels="inferred",
            label_mode="binary",
            image_size=image_size,
            batch_size=self.config.batch_size,
            shuffle=False,
        )
        self.class_names = test_ds.class_names
        print("Detected class names:", self.class_names)
        return test_ds

    def configure_performance(self, test_ds):
        autotune = tf.data.AUTOTUNE
        test_ds = test_ds.cache().prefetch(buffer_size=autotune)
        return test_ds


class ModelLocator:
    """Finds the most recent saved model file in the weights directory."""

    def __init__(self, weights_dir: str):
        self.weights_dir = weights_dir

    def find_latest_model(self) -> str:
        if not os.path.exists(self.weights_dir):
            raise FileNotFoundError("Weights directory not found: " + self.weights_dir)

        candidates = []
        for fname in os.listdir(self.weights_dir):
            if fname.endswith(".keras"):
                fpath = os.path.join(self.weights_dir, fname)
                candidates.append((os.path.getmtime(fpath), fpath))

        if not candidates:
            raise FileNotFoundError("No model files with .keras extension found in: " + self.weights_dir)

        candidates.sort(key=lambda x: x[0], reverse=True)
        return candidates[0][1]


class Evaluator:
    """Runs model evaluation, predictions, and produces analysis outputs."""

    def __init__(self, model: keras.Model, config: EvaluationConfig, class_names: list):
        self.model = model
        self.config = config
        self.class_names = class_names

    def evaluate(self, test_ds) -> dict:
        results = self.model.evaluate(test_ds, verbose=1)
        metrics = dict(zip(self.model.metrics_names, results))
        return metrics

    def predict(self, test_ds):
        y_prob = self.model.predict(test_ds, verbose=1).reshape(-1)
        y_pred = (y_prob >= self.config.threshold).astype(int)

        y_true_list = []
        for _, labels in test_ds:
            y_true_list.append(labels.numpy().reshape(-1))
        y_true = np.concatenate(y_true_list, axis=0).astype(int)

        return y_true, y_prob, y_pred

    def compute_confusion_matrix(self, y_true, y_pred):
        return confusion_matrix(y_true, y_pred)

    def compute_classification_report(self, y_true, y_pred) -> str:
        target_names = self.class_names
        report = classification_report(y_true, y_pred, target_names=target_names, digits=4)
        return report

    def compute_auc(self, y_true, y_prob) -> float:
        try:
            return float(roc_auc_score(y_true, y_prob))
        except Exception:
            return float("nan")


class ArtifactWriter:
    """Saves evaluation artifacts to disk."""

    def __init__(self, artifact_paths: dict):
        self.artifact_paths = artifact_paths

    def save_model_used(self, model_path: str) -> None:
        with open(self.artifact_paths["model_used_path"], "w") as f:
            f.write(model_path)
        print("Model used saved to:", self.artifact_paths["model_used_path"])

    def save_predictions(self, file_paths: list, y_true, y_prob, y_pred) -> None:
        df = pd.DataFrame({
            "filepath": file_paths,
            "y_true": y_true,
            "y_prob": y_prob,
            "y_pred": y_pred,
        })
        df.to_csv(self.artifact_paths["predictions_csv_path"], index=False)
        print("Predictions saved to:", self.artifact_paths["predictions_csv_path"])

    def save_confusion_matrix(self, cm: np.ndarray, class_names: list) -> None:
        df = pd.DataFrame(cm, index=class_names, columns=class_names)
        df.to_csv(self.artifact_paths["confusion_matrix_csv_path"], index=True)
        print("Confusion matrix saved to:", self.artifact_paths["confusion_matrix_csv_path"])

    def save_metrics_json(self, metrics: dict) -> None:
        with open(self.artifact_paths["metrics_json_path"], "w") as f:
            json.dump(metrics, f, indent=2)
        print("Metrics saved to:", self.artifact_paths["metrics_json_path"])

    def save_classification_report(self, report_text: str) -> None:
        with open(self.artifact_paths["classification_report_path"], "w") as f:
            f.write(report_text)
        print("Classification report saved to:", self.artifact_paths["classification_report_path"])


class FigureWriter:
    """Creates and saves plots used for evaluation."""

    def __init__(self, artifact_paths: dict):
        self.artifact_paths = artifact_paths

    def plot_confusion_matrix(self, cm: np.ndarray, class_names: list) -> None:
        fig = plt.figure(figsize=(6, 5))
        plt.imshow(cm, interpolation="nearest")
        plt.title("Confusion Matrix")
        plt.colorbar()
        tick_marks = np.arange(len(class_names))
        plt.xticks(tick_marks, class_names, rotation=45)
        plt.yticks(tick_marks, class_names)

        thresh = cm.max() / 2.0 if cm.max() > 0 else 0.0
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                color = "white" if cm[i, j] > thresh else "black"
                plt.text(j, i, format(cm[i, j], "d"), horizontalalignment="center", color=color)

        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.savefig(self.artifact_paths["confusion_matrix_plot_path"], dpi=200)
        plt.show()
        print("Confusion matrix plot saved to:", self.artifact_paths["confusion_matrix_plot_path"])

    def plot_error_grid(self, images: np.ndarray, titles: list) -> None:
        if images is None or len(images) == 0:
            print("No error samples available for plotting")
            return

        n = len(images)
        cols = 6
        rows = int(np.ceil(n / cols))

        fig = plt.figure(figsize=(cols * 3, rows * 3))
        for idx in range(n):
            ax = fig.add_subplot(rows, cols, idx + 1)
            ax.imshow(images[idx].astype(np.uint8))
            ax.set_title(titles[idx], fontsize=8)
            ax.axis("off")

        plt.tight_layout()
        plt.savefig(self.artifact_paths["error_grid_plot_path"], dpi=200)
        plt.show()
        print("Error sample grid saved to:", self.artifact_paths["error_grid_plot_path"])


In [ ]:
class ErrorSampler:
    """Collects a limited set of misclassified samples for qualitative analysis."""

    def __init__(self, config: EvaluationConfig):
        self.config = config

    def collect(self, test_ds, y_true, y_prob, y_pred):
        error_indices = np.where(y_true != y_pred)[0]
        if len(error_indices) == 0:
            return None, []

        k = min(self.config.max_error_samples, len(error_indices))
        selected = error_indices[:k]

        images_collected = []
        titles = []

        flat_images = []
        flat_labels = []
        for batch_images, batch_labels in test_ds:
            flat_images.append(batch_images.numpy())
            flat_labels.append(batch_labels.numpy().reshape(-1))
        flat_images = np.concatenate(flat_images, axis=0)
        flat_labels = np.concatenate(flat_labels, axis=0).astype(int)

        for idx in selected:
            img = flat_images[idx]
            img_uint8 = np.clip(img, 0, 255).astype(np.uint8)

            true_label = int(y_true[idx])
            pred_label = int(y_pred[idx])
            prob = float(y_prob[idx])

            title = f"true {true_label} pred {pred_label} prob {prob:.3f}"
            images_collected.append(img_uint8)
            titles.append(title)

        return np.array(images_collected), titles


In [ ]:
def extract_filepaths_from_directory(test_dir: str) -> list:
    """Extracts file paths in the same order used by image_dataset_from_directory with shuffle disabled."""

    filepaths = []
    for class_name in sorted(os.listdir(test_dir)):
        class_path = os.path.join(test_dir, class_name)
        if not os.path.isdir(class_path):
            continue
        for fname in sorted(os.listdir(class_path)):
            fpath = os.path.join(class_path, fname)
            if os.path.isfile(fpath):
                filepaths.append(fpath)
    return filepaths


In [ ]:
def main() -> None:
    drive_manager = DriveManager()
    drive_manager.mount_drive(force_remount=False)

    project = ProjectConfig(base_dir="/content/drive/MyDrive/PatchParkCNN")
    project.ensure_directories()
    project.assert_test_structure()
    artifacts = project.artifact_paths()

    inspector = SystemInspector()
    inspector.print_device_summary()

    config = EvaluationConfig()

    model_path = ModelLocator(project.weights_dir).find_latest_model()
    model = keras.models.load_model(model_path)
    print("Loaded model:", model_path)

    test_ds = DatasetLoader(project.test_dir, config).load()
    test_ds = DatasetLoader(project.test_dir, config).configure_performance(test_ds)

    class_names = sorted([d for d in os.listdir(project.test_dir) if os.path.isdir(os.path.join(project.test_dir, d))])

    evaluator = Evaluator(model=model, config=config, class_names=class_names)
    eval_metrics = evaluator.evaluate(test_ds)

    y_true, y_prob, y_pred = evaluator.predict(test_ds)
    cm = evaluator.compute_confusion_matrix(y_true, y_pred)
    report_text = evaluator.compute_classification_report(y_true, y_pred)
    auc_value = evaluator.compute_auc(y_true, y_prob)

    summary_metrics = dict(eval_metrics)
    summary_metrics["auc_score"] = auc_value
    summary_metrics["threshold"] = config.threshold

    print("Evaluation metrics:", summary_metrics)
    print("Classification report:")
    print(report_text)

    file_paths = extract_filepaths_from_directory(project.test_dir)
    min_len = min(len(file_paths), len(y_true))
    file_paths = file_paths[:min_len]
    y_true = y_true[:min_len]
    y_prob = y_prob[:min_len]
    y_pred = y_pred[:min_len]

    writer = ArtifactWriter(artifact_paths=artifacts)
    writer.save_model_used(model_path)
    writer.save_predictions(file_paths, y_true, y_prob, y_pred)
    writer.save_confusion_matrix(cm, class_names)
    writer.save_metrics_json(summary_metrics)
    writer.save_classification_report(report_text)

    fig_writer = FigureWriter(artifact_paths=artifacts)
    fig_writer.plot_confusion_matrix(cm, class_names)

    sampler = ErrorSampler(config)
    error_images, error_titles = sampler.collect(test_ds, y_true, y_prob, y_pred)
    fig_writer.plot_error_grid(error_images, error_titles)

    print("Notebook 6 completed successfully")


if __name__ == "__main__":
    main()


#07 Grad CAM Interpretability
PatchParkCNN Project
Advanced Mathematical Concepts for Deep Learning

We apply Grad CAM interpretability to a trained convolutional neural network in order to visualize which spatial regions of parking slot images contribute most to the free versus busy classification decisions.

In [ ]:
from google.colab import drive
import os
import time
import json
import numpy as np
import matplotlib.pyplot as plt
import cv2


import tensorflow as tf
from tensorflow import keras


In [ ]:
class DriveManager:
    """Handles Google Drive mounting in Colab."""

    def __init__(self, mount_point: str = "/content/drive"):
        self.mount_point = mount_point

    def mount_drive(self, force_remount: bool = False) -> None:
        drive.mount(self.mount_point, force_remount=force_remount)
        print("Drive mounted at:", self.mount_point)


In [ ]:
class ProjectConfig:
    """Centralizes paths used for Grad CAM analysis."""

    def __init__(self, base_dir: str = "/content/drive/MyDrive/PatchParkCNN"):
        self.base_dir = base_dir

        self.data_dir = os.path.join(self.base_dir, "data_patches_cnr", "test")
        self.weights_dir = os.path.join(self.base_dir, "weights")
        self.figures_dir = os.path.join(self.base_dir, "figures")
        self.report_dir = os.path.join(self.base_dir, "report")

        self.run_id = time.strftime("%Y%m%d_%H%M%S")

    def ensure_directories(self) -> None:
        for path in [self.figures_dir, self.report_dir]:
            os.makedirs(path, exist_ok=True)
        print("Project directories are ready")

    def artifact_paths(self) -> dict:
        return {
            "gradcam_grid_path": os.path.join(self.figures_dir, f"gradcam_grid_{self.run_id}.png"),
            "gradcam_log_path": os.path.join(self.report_dir, f"gradcam_notes_{self.run_id}.txt"),
        }


In [ ]:
class ModelLocator:
    """Finds the most recent trained model."""

    def __init__(self, weights_dir: str):
        self.weights_dir = weights_dir

    def find_latest_model(self) -> str:
        candidates = []
        for fname in os.listdir(self.weights_dir):
            if fname.endswith(".keras"):
                fpath = os.path.join(self.weights_dir, fname)
                candidates.append((os.path.getmtime(fpath), fpath))
        if not candidates:
            raise FileNotFoundError("No trained model found")
        candidates.sort(key=lambda x: x[0], reverse=True)
        return candidates[0][1]


In [ ]:
class ImageSampler:
    """Samples representative images from test set for Grad CAM analysis."""

    def __init__(self, data_dir: str, max_samples: int = 12):
        self.data_dir = data_dir
        self.max_samples = max_samples

    def sample_images(self):
        samples = []
        labels = []
        class_names = sorted(os.listdir(self.data_dir))

        for class_name in class_names:
            class_path = os.path.join(self.data_dir, class_name)
            files = sorted(os.listdir(class_path))[: self.max_samples // 2]
            for fname in files:
                samples.append(os.path.join(class_path, fname))
                labels.append(class_name)
        return samples, labels


In [ ]:
class GradCAM:
    """Implements Grad CAM for TensorFlow Keras models."""

    def __init__(self, model: keras.Model, last_conv_layer_name: str):
        self.model = model
        self.last_conv_layer_name = last_conv_layer_name

        self.grad_model = keras.models.Model(
            [self.model.inputs],
            [self.model.get_layer(self.last_conv_layer_name).output, self.model.output],
        )

    def compute_heatmap(self, image_tensor):
        with tf.GradientTape() as tape:
            conv_outputs, predictions = self.grad_model(image_tensor)
            loss = predictions[:, 0]

        grads = tape.gradient(loss, conv_outputs)
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

        conv_outputs = conv_outputs[0]
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)

        heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
        return heatmap.numpy()


In [ ]:
class GradCAMVisualizer:
    """Overlays Grad CAM heatmaps on original images."""

    def overlay(self, image, heatmap, alpha: float = 0.4):
        heatmap = np.uint8(255 * heatmap)
        heatmap = np.stack([heatmap] * 3, axis=-1)
        overlay = heatmap * alpha + image
        overlay = np.clip(overlay, 0, 255).astype(np.uint8)
        return overlay


In [ ]:
def load_and_preprocess_image(path, image_size=(128, 128)):
    img = keras.utils.load_img(path, target_size=image_size)
    img_array = keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0
    return img_array, img


In [ ]:

def main() -> None:
    drive_manager = DriveManager()
    drive_manager.mount_drive()

    project = ProjectConfig()
    project.ensure_directories()
    artifacts = project.artifact_paths()

    model_path = ModelLocator(project.weights_dir).find_latest_model()
    model = keras.models.load_model(model_path)
    print("Loaded model:", model_path)

    # Automatically select last convolutional layer
    last_conv_layer_name = None
    for layer in reversed(model.layers):
        if isinstance(layer, keras.layers.Conv2D):
            last_conv_layer_name = layer.name
            break

    if last_conv_layer_name is None:
        raise RuntimeError("No convolutional layer found in model")

    print("Using last convolutional layer:", last_conv_layer_name)

    sampler = ImageSampler(project.data_dir)
    image_paths, labels = sampler.sample_images()

    gradcam = GradCAM(model, last_conv_layer_name)
    visualizer = GradCAMVisualizer()

    fig = plt.figure(figsize=(12, 8))
    for idx, path in enumerate(image_paths):
        img_tensor, img_pil = load_and_preprocess_image(path)
        original_image_array = np.array(img_pil) # Store original image as array
        heatmap = gradcam.compute_heatmap(img_tensor)

        # Resize heatmap to match the original image dimensions
        h, w, _ = original_image_array.shape
        resized_heatmap = cv2.resize(heatmap, (w, h), interpolation=cv2.INTER_LINEAR)

        overlay = visualizer.overlay(original_image_array, resized_heatmap)

        ax = fig.add_subplot(3, 4, idx + 1)
        ax.imshow(overlay)
        ax.set_title(labels[idx])
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(artifacts["gradcam_grid_path"], dpi=200)
    plt.show()

    with open(artifacts["gradcam_log_path"], "w") as f:
        f.write("Grad CAM analysis completed using layer: " + last_conv_layer_name + "")
        f.write("Model path: " + model_path + "")

    print("Grad CAM figure saved to:", artifacts["gradcam_grid_path"])
    print("Notebook 7 completed successfully")


if __name__ == "__main__":
    main()


#08 Streamlit Mini App
PatchParkCNN Project
Advanced Mathematical Concepts for Deep Learning

We build a lightweight Streamlit application that loads the trained CNN model and performs single image inference for parking slot classification (free vs busy). The app is intended for demonstration and stakeholder communication.

In [ ]:
from google.colab import drive
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow import keras


In [ ]:
class DriveManager:
    """Handles Google Drive mounting in Colab."""

    def __init__(self, mount_point: str = "/content/drive"):
        self.mount_point = mount_point

    def mount_drive(self, force_remount: bool = False) -> None:
        drive.mount(self.mount_point, force_remount=force_remount)
        print("Drive mounted at:", self.mount_point)


In [ ]:
class ProjectConfig:
    """Centralizes paths used by the Streamlit mini app."""

    def __init__(self, base_dir: str = "/content/drive/MyDrive/PatchParkCNN"):
        self.base_dir = base_dir
        self.weights_dir = os.path.join(self.base_dir, "weights")
        self.app_dir = os.path.join(self.base_dir, "streamlit_app")

    def ensure_directories(self) -> None:
        os.makedirs(self.app_dir, exist_ok=True)
        print("Streamlit app directory:", self.app_dir)

    def app_files(self) -> dict:
        return {
            "app_py": os.path.join(self.app_dir, "app.py"),
            "requirements_txt": os.path.join(self.app_dir, "requirements.txt"),
            "readme_md": os.path.join(self.app_dir, "README.md"),
        }


In [ ]:
class ModelLocator:
    """Finds the most recent trained model."""

    def __init__(self, weights_dir: str):
        self.weights_dir = weights_dir

    def find_latest_model(self) -> str:
        candidates = []
        for fname in os.listdir(self.weights_dir):
            if fname.endswith(".keras"):
                fpath = os.path.join(self.weights_dir, fname)
                candidates.append((os.path.getmtime(fpath), fpath))
        if not candidates:
            raise FileNotFoundError("No trained model found")
        candidates.sort(key=lambda x: x[0], reverse=True)
        return candidates[0][1]


In [ ]:
class StreamlitAppWriter:
    """Writes a minimal Streamlit app for single image inference."""

    def __init__(self, model_path: str, app_files: dict):
        self.model_path = model_path
        self.app_files = app_files

    def write_app(self) -> None:
        # The f-string for app_code. Note: double curly braces to escape expressions
        # intended for the *generated* app's f-strings.
        app_code = f"""
import streamlit as st
import numpy as np
from tensorflow import keras
from PIL import Image

st.set_page_config(page_title='PatchParkCNN', layout='centered')

st.title('PatchParkCNN Mini App')
st.write('Single image inference for parking slot classification')

@st.cache_resource
def load_model():
    return keras.models.load_model(r'{self.model_path}')

model = load_model()

uploaded_file = st.file_uploader(
    'Upload a parking slot image',
    type=['jpg', 'png', 'jpeg']
)

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert('RGB')
    st.image(image, caption='Uploaded image', use_column_width=True)

    img = image.resize((128, 128))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    prob_free = float(model.predict(img_array, verbose=0)[0][0])
    pred_label = "free" if prob_free >= 0.5 else "busy"

    st.subheader("Prediction")
    st.write(f"Class: **{{pred_label}}**") # Escaped f-string for generated app
    st.write(f"Probability (free): {{prob_free:.3f}}") # Escaped f-string for generated app
"""

        with open(self.app_files["app_py"], "w") as f:
            f.write(app_code)

    def write_requirements(self) -> None:
        requirements = "\n".join(["streamlit", "tensorflow", "pillow", "numpy"])
        with open(self.app_files["requirements_txt"], "w") as f:
            f.write(requirements)

    def write_readme(self) -> None:
        text = (
            "PatchParkCNN Streamlit Mini App\n\n"
            "Run locally:\n"
            "streamlit run app.py\n"
        )
        with open(self.app_files["readme_md"], "w") as f:
            f.write(text)

In [ ]:
def main() -> None:
    drive_manager = DriveManager()
    drive_manager.mount_drive(force_remount=False)

    project = ProjectConfig()
    project.ensure_directories()

    model_path = ModelLocator(project.weights_dir).find_latest_model()
    print('Using model:', model_path)

    app_files = project.app_files()
    writer = StreamlitAppWriter(model_path=model_path, app_files=app_files)
    writer.write_app()
    writer.write_requirements()
    writer.write_readme()

    print('Streamlit mini app generated')
    for k, v in app_files.items():
        print(k, ':', v)


if __name__ == '__main__':
    main()
